# Hull-White Model

The Hull-White model extends the Vasicek framework by incorporating time-dependent parameters to fit the initial term structure of interest rates exactly. Introduced by John Hull and Alan White in the early 1990s, this model remains a canonical one-factor short-rate model for pricing interest-rate derivatives and fixed-income instruments within a risk-neutral valuation context.

## 1. Fundamental Assumptions

The Hull-White model preserves many classical assumptions of short-rate modeling while enhancing empirical flexibility:

* **Single Stochastic Driver:** The instantaneous short rate is the only source of randomness.
* **Mean Reversion:** The short rate reverts toward a time-dependent target level, allowing for a more accurate representation of the evolving term structure.
* **Gaussian Dynamics:** The short rate follows a Gaussian diffusion process, permitting analytically tractable bond pricing formulas.
* **Time-Dependent Parameters:** The drift term includes a deterministic function calibrated to the observed initial yield curve.
* **Frictionless Market:** Markets are assumed frictionless, with continuous trading, no transaction costs, and absence of arbitrage.

## 2. Key Model Parameters

The model specification requires:

1. $r_t$: short-term instantaneous interest rate at time $t$.
2. $a$: constant speed of mean reversion.
3. $\sigma$: constant volatility parameter.
4. $\theta(t)$: deterministic time-dependent mean reversion level.
5. $T - t$: time to maturity for the instrument being valued.

## 3. Mathematical Formulation

The Hull-White one-factor short-rate dynamics are described by the stochastic differential equation:

$$
dr_t = \left[ \theta(t) - a r_t \right] dt + \sigma \, dW_t
$$

where $W_t$ denotes a standard Brownian motion under the risk-neutral measure. The process is an Ornstein-Uhlenbeck type diffusion with a time-dependent drift term.

Key properties include:

* mean reversion: $\theta(t) - a r_t$
* diffusion: $\sigma dW_t$
* exact fit to initial term structure through $\theta(t)$

## 4. Analytical Bond Pricing

Under the Hull-White model, the zero-coupon bond price $P(t,T)$ can be written in exponential-affine form:

$$
P(t,T) = \exp\left[ A(t,T) - B(t,T) r_t \right]
$$

with

$$
B(t,T) = \frac{1 - e^{-a(T-t)}}{a}
$$

and

$$
A(t,T) = \ln\frac{P(0,T)}{P(0,t)} + B(t,T) f(0,t) - \frac{\sigma^{2}}{4a} B(t,T)^{2}
$$

where $f(0,t)$ is the initial forward rate curve, and $P(0,t)$ is the initial zero-coupon bond price at time $0$. The function $\theta(t)$ is calibrated to satisfy:

$$
\theta(t) = \frac{\partial f(0,t)}{\partial t} + a f(0,t) + \frac{\sigma^{2}}{2a} \left(1 - e^{-2 a t}\right)
$$

This calibration ensures that the model reproduces the observed initial yield curve exactly.

## 5. Interpretation and Use

The Hull-White model is valued for its combination of analytical tractability and flexibility:

* it retains closed-form expressions for bond prices and certain interest-rate derivatives;
* it accommodates the term structure through a deterministic shift function $\theta(t)$;
* it provides a foundation for pricing European caps, floors, and swaptions in a one-factor Gaussian setting.

## 6. Limitations

Despite its practical appeal, the Hull-White model has theoretical limitations:

* it allows negative short rates due to the Gaussian assumption;
* it relies on a single risk factor, ignoring multiple sources of term structure risk;
* the constant volatility assumption may not capture state-dependent market dynamics.

In summary, the Hull-White model is a pedagogical and practical tool for interest-rate modeling, particularly useful when an exact fit to the initial yield curve is required while preserving analytical convenience.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hull-White parameters
a = 0.1
sigma = 0.01
r0 = 0.03
f0 = 0.03  # flat initial forward curve

def theta(t):
    return a * f0 + sigma**2 / (2 * a) * (1 - np.exp(-2 * a * t))

def B(t, T):
    return (1 - np.exp(-a * (T - t))) / a

def A(t, T):
    return -f0 * (T - t) + B(t, T) * f0 - (sigma**2 / (4 * a)) * B(t, T)**2

def P(t, T, rt):
    return np.exp(A(t, T) - B(t, T) * rt)

def simulate_hull_white(n_paths=10, T=5.0, dt=0.01):
    n_steps = int(T / dt)
    ts = np.linspace(0, T, n_steps + 1)
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = r0
    for i in range(n_steps):
        t = ts[i]
        dW = np.sqrt(dt) * np.random.randn(n_paths)
        paths[:, i + 1] = paths[:, i] + (theta(t) - a * paths[:, i]) * dt + sigma * dW
    return ts, paths

ts, paths = simulate_hull_white(n_paths=12, T=5.0, dt=0.01)

plt.figure(figsize=(10, 5))
for i in range(paths.shape[0]):
    plt.plot(ts, paths[i], alpha=0.8)
plt.title("Hull-White Short Rate Simulation")
plt.xlabel("Time")
plt.ylabel("Short rate $r_t$")
plt.grid(True)
plt.show()

T_grid = np.linspace(0.1, 10, 100)
P0 = [P(0, T, r0) for T in T_grid]

plt.figure(figsize=(8, 4))
plt.plot(T_grid, P0, label="$P(0,T)$")
plt.title("Zero-Coupon Bond Prices under Hull-White")
plt.xlabel("Maturity $T$")
plt.ylabel("Bond price")
plt.grid(True)
plt.legend()
plt.show()